# Forecasting de ventas
Este notebook se utilizar? para generar predicciones de ventas a partir de nuevos datos de inferencia. El objetivo es replicar las transformaciones del notebook de entrenamiento, dejar preparado `inferencia_df` con la misma estructura de entrada del modelo final y conservar ?nicamente los registros de noviembre para la inferencia.


In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
from pathlib import Path
import holidays
import joblib


In [23]:
ruta_inferencia = Path('../data/raw/inferencia/ventas_2025_inferencia.csv')
inferencia_2025 = pd.read_csv(ruta_inferencia)

inferencia_2025.head()


,fecha,producto_id,nombre,categoria,subcategoria,precio_base,es_estrella,unidades_vendidas,precio_venta,ingresos,Amazon,Decathlon,Deporvillage
0,2025-10-25,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,26.0,113.13,2941.38,89.51,113.43,104.78
1,2025-10-25,PROD_002,Adidas Ultraboost 23,Running,Zapatillas Running,135,True,27.0,141.89,3831.03,128.73,112.91,122.88
2,2025-10-25,PROD_003,Asics Gel Nimbus 25,Running,Zapatillas Running,85,False,5.0,85.79,428.95,84.28,74.51,85.57
3,2025-10-25,PROD_004,New Balance Fresh Foam X 1080v12,Running,Zapatillas Running,75,False,3.0,76.19,228.57,75.54,70.32,71.13
4,2025-10-25,PROD_005,Nike Dri-FIT Miler,Running,Ropa Running,35,False,3.0,35.48,106.44,33.84,31.32,34.41


In [24]:
ruta_modelo = Path('models/modelofinal.joblib')
modelo_final = joblib.load(ruta_modelo)

print('Modelo final cargado correctamente.')
print(f'Numero de variables esperadas por el modelo: {len(modelo_final.feature_names_in_)}')


Modelo final cargado correctamente.
Numero de variables esperadas por el modelo: 440


### Diferencia entre `inferencia_df` e `inferencia_df_modelo`
En este notebook usamos dos dataframes con funciones distintas.

`inferencia_df` es el dataframe transformado y legible para an?lisis. Conserva columnas descriptivas como `fecha`, `nombre`, `categoria` o `nombre_festivo`, que son ?tiles para revisar los datos, validarlos y presentar resultados.

`inferencia_df_modelo` es el dataframe final que entra realmente al modelo. Se construye a partir de `inferencia_df`, pero se alinea exactamente con las variables que espera `modelo_final` en `modelo_final.feature_names_in_`.

Esto significa que:
- si falta alguna columna one hot o alguna variable del entrenamiento, se crea con valor `0`
- solo se conservan las columnas exactas que el modelo necesita
- el orden de las columnas queda igual que en el entrenamiento final

En resumen:
- `inferencia_df` sirve para inspecci?n, trazabilidad y guardado del dataset transformado
- `inferencia_df_modelo` sirve para generar predicciones sin errores de estructura
- la app y el modelo deben usar la versi?n alineada al modelo, es decir, `inferencia_df_modelo`


In [25]:
# Nota importante:
# `inferencia_df` es el dataframe transformado y legible para revisi?n.
# `inferencia_df_modelo` es el que entra realmente al modelo porque queda
# alineado exactamente con `modelo_final.feature_names_in_`.
#
# Por eso, aunque `inferencia_df` no muestre todas las columnas del entrenamiento,
# `inferencia_df_modelo` s? contiene la estructura final que necesita el modelo.
features_modelo_final = list(modelo_final.feature_names_in_)
for col in features_modelo_final:
    if col not in inferencia_df.columns:
        inferencia_df[col] = 0

inferencia_df_modelo = inferencia_df[features_modelo_final].copy()


Forma final de inferencia_df: (720, 98)
Columnas finales de inferencia_df:
['fecha', 'producto_id', 'nombre', 'categoria', 'subcategoria', 'precio_base', 'es_estrella', 'unidades_vendidas', 'precio_venta', 'ingresos', 'anio', 'trimestre', 'mes', 'nombre_mes', 'dia', 'dia_anio', 'semana_anio', 'dia_semana_num', 'dia_semana', 'es_fin_de_semana', 'es_inicio_mes', 'es_fin_mes', 'es_inicio_trimestre', 'es_fin_trimestre', 'es_inicio_anio', 'es_fin_anio', 'es_primera_quincena', 'es_ultima_quincena', 'es_payday_inicio_mes', 'es_payday_fin_mes', 'es_festivo', 'nombre_festivo', 'es_vispera_festivo', 'es_post_festivo', 'es_black_friday', 'es_cyber_monday', 'dias_hasta_fin_mes', 'semana_mes', 'temporada_rebajas_invierno', 'temporada_rebajas_verano', 'campana_navidad', 'campana_vuelta_al_cole', 'es_puente', 'lag_1', 'lag_2', 'lag_3', 'lag_4', 'lag_5', 'lag_6', 'lag_7', 'media_movil_7_dias', 'descuento_porcentaje', 'precio_competencia', 'ratio_precio', 'nombre_h_Adidas Own The Run Jacket', 'nombre_h

lag_1                 696
media_movil_7_dias    696
lag_2                 672
lag_3                 648
lag_4                 624
lag_5                 600
lag_6                 576
lag_7                 552
dtype: int64

,fecha,producto_id,nombre,unidades_vendidas
168,2025-11-01,PROD_001,Nike Air Zoom Pegasus 40,NaN
192,2025-11-02,PROD_001,Nike Air Zoom Pegasus 40,NaN
216,2025-11-03,PROD_001,Nike Air Zoom Pegasus 40,NaN
240,2025-11-04,PROD_001,Nike Air Zoom Pegasus 40,NaN
264,2025-11-05,PROD_001,Nike Air Zoom Pegasus 40,NaN


In [26]:
ruta_salida = Path('../data/processed/inferencia_df_transformado.csv')
ruta_salida.parent.mkdir(parents=True, exist_ok=True)
inferencia_df.to_csv(ruta_salida, index=False)

print(f'DataFrame transformado guardado en: {ruta_salida}')


DataFrame transformado guardado en: ..\data\processed\inferencia_df_transformado.csv


In [39]:
# Mostramos la forma final del DataFrame transformado.
print('Forma final de inferencia_df:', inferencia_df.shape)

# Mostramos todas las columnas por bloques para evitar que Jupyter recorte la salida.
columnas = inferencia_df.columns.tolist()
tamano_bloque = 20

for inicio in range(0, len(columnas), tamano_bloque):
    fin = inicio + tamano_bloque
    bloque = pd.DataFrame({'columna': columnas[inicio:fin]}, index=range(inicio + 1, min(fin, len(columnas)) + 1))
    display(bloque)

# Guardamos el DataFrame transformado para reutilizarlo en la inferencia.
ruta_salida = Path('../data/processed/inferencia_df_transformado.csv')
ruta_salida.parent.mkdir(parents=True, exist_ok=True)
inferencia_df.to_csv(ruta_salida, index=False)

print(f'DataFrame guardado en: {ruta_salida}')


Forma final de inferencia_df: (720, 98)


,columna
1,fecha
2,producto_id
3,nombre
4,categoria
5,subcategoria
6,precio_base
7,es_estrella
8,unidades_vendidas
9,precio_venta
10,ingresos


,columna
21,es_inicio_mes
22,es_fin_mes
23,es_inicio_trimestre
24,es_fin_trimestre
25,es_inicio_anio
26,es_fin_anio
27,es_primera_quincena
28,es_ultima_quincena
29,es_payday_inicio_mes
30,es_payday_fin_mes


,columna
41,campana_navidad
42,campana_vuelta_al_cole
43,es_puente
44,lag_1
45,lag_2
46,lag_3
47,lag_4
48,lag_5
49,lag_6
50,lag_7


,columna
61,nombre_h_Domyos BM900
62,nombre_h_Domyos Kit Mancuernas 20kg
63,nombre_h_Gaiam Premium Yoga Block
64,nombre_h_Liforme Yoga Pad
65,nombre_h_Lotuscrafts Yoga Bolster
66,nombre_h_Manduka PRO Yoga Mat
67,nombre_h_Merrell Moab 2 GTX
68,nombre_h_New Balance Fresh Foam X 1080v12
69,nombre_h_Nike Air Zoom Pegasus 40
70,nombre_h_Nike Dri-FIT Miler


,columna
81,categoria_h_Running
82,categoria_h_Wellness
83,subcategoria_h_Banco Gimnasio
84,subcategoria_h_Bandas Elásticas
85,subcategoria_h_Bicicleta Montaña
86,subcategoria_h_Bloque Yoga
87,subcategoria_h_Cojín Yoga
88,subcategoria_h_Esterilla Fitness
89,subcategoria_h_Esterilla Yoga
90,subcategoria_h_Mancuernas Ajustables


DataFrame guardado en: ..\data\processed\inferencia_df_transformado.csv


In [40]:
inferencia_df.head()

,fecha,producto_id,nombre,categoria,subcategoria,precio_base,es_estrella,unidades_vendidas,precio_venta,ingresos,anio,trimestre,mes,nombre_mes,dia,dia_anio,semana_anio,dia_semana_num,dia_semana,es_fin_de_semana,es_inicio_mes,es_fin_mes,es_inicio_trimestre,es_fin_trimestre,es_inicio_anio,es_fin_anio,es_primera_quincena,es_ultima_quincena,es_payday_inicio_mes,es_payday_fin_mes,es_festivo,nombre_festivo,es_vispera_festivo,es_post_festivo,es_black_friday,es_cyber_monday,dias_hasta_fin_mes,semana_mes,temporada_rebajas_invierno,temporada_rebajas_verano,campana_navidad,campana_vuelta_al_cole,es_puente,lag_1,lag_2,lag_3,lag_4,lag_5,lag_6,lag_7,media_movil_7_dias,descuento_porcentaje,precio_competencia,ratio_precio,nombre_h_Adidas Own The Run Jacket,nombre_h_Adidas Ultraboost 23,nombre_h_Asics Gel Nimbus 25,nombre_h_Bowflex SelectTech 552,nombre_h_Columbia Silver Ridge,nombre_h_Decathlon Bandas Elásticas Set,nombre_h_Domyos BM900,nombre_h_Domyos Kit Mancuernas 20kg,nombre_h_Gaiam Premium Yoga Block,nombre_h_Liforme Yoga Pad,nombre_h_Lotuscrafts Yoga Bolster,nombre_h_Manduka PRO Yoga Mat,nombre_h_Merrell Moab 2 GTX,nombre_h_New Balance Fresh Foam X 1080v12,nombre_h_Nike Air Zoom Pegasus 40,nombre_h_Nike Dri-FIT Miler,nombre_h_Puma Velocity Nitro 2,nombre_h_Quechua MH500,nombre_h_Reebok Floatride Energy 5,nombre_h_Reebok Professional Deck,nombre_h_Salomon Speedcross 5 GTX,nombre_h_Sveltus Kettlebell 12kg,nombre_h_The North Face Borealis,nombre_h_Trek Marlin 7,categoria_h_Fitness,categoria_h_Outdoor,categoria_h_Running,categoria_h_Wellness,subcategoria_h_Banco Gimnasio,subcategoria_h_Bandas Elásticas,subcategoria_h_Bicicleta Montaña,subcategoria_h_Bloque Yoga,subcategoria_h_Cojín Yoga,subcategoria_h_Esterilla Fitness,subcategoria_h_Esterilla Yoga,subcategoria_h_Mancuernas Ajustables,subcategoria_h_Mochila Trekking,subcategoria_h_Pesa Rusa,subcategoria_h_Pesas Casa,subcategoria_h_Rodillera Yoga,subcategoria_h_Ropa Montaña,subcategoria_h_Ropa Running,subcategoria_h_Zapatillas Running,subcategoria_h_Zapatillas Trail
168,2025-11-01,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,NaN,115.0,NaN,2025,4,11,November,1,305,44,5,Saturday,True,True,False,False,False,False,False,True,False,True,False,True,Todos los Santos,False,False,False,False,29,1,False,False,True,False,False,14.0,10.0,14.0,15.0,16.0,20.0,26.0,16.428571,0.0,100.610000,1.143028,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
192,2025-11-02,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,NaN,115.0,NaN,2025,4,11,November,2,306,44,6,Sunday,True,False,False,False,False,False,False,True,False,True,False,False,None,False,True,False,False,28,1,False,False,True,False,True,NaN,14.0,10.0,14.0,15.0,16.0,20.0,NaN,0.0,97.740000,1.176591,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
216,2025-11-03,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,NaN,115.0,NaN,2025,4,11,November,3,307,45,0,Monday,False,False,False,False,False,False,False,True,False,True,False,False,None,False,False,False,False,27,1,False,False,True,False,False,NaN,NaN,14.0,10.0,14.0,15.0,16.0,NaN,0.0,99.513333,1.155624,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
240,2025-11-04,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,NaN,115.0,NaN,2025,4,11,November,4,308,45,1,Tuesday,False,False,False,False,False,False,False,True,False,True,False,False,None,False,False,False,False,26,1,False,False,True,False,False,NaN,NaN,NaN,14.0,10.0,14.0,15.0,NaN,0.0,105.553333,1.089497,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
264,2025-11-05,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,NaN,115.0,NaN,2025,4,11,November,5,309,45,2,Wednesday,False,False,False,False,False,False,False,True,False,True,False,False,None,False,False,False,False,25,1,False,False,True,False,False,NaN,

In [ ]:
# Objetivo: verificar que las variables usadas por el modelo final est?n completas en inferencia.
# Nota: `inferencia_df` puede tener menos columnas visibles porque conserva campos descriptivos,
# pero `inferencia_df_modelo` es el DataFrame realmente alineado con las variables del modelo.
features_modelo_final = list(modelo_final.feature_names_in_)
columnas_inferencia = inferencia_df.columns.tolist()
columnas_modelo = inferencia_df_modelo.columns.tolist()

faltan_en_inferencia_df = [col for col in features_modelo_final if col not in columnas_inferencia]
faltan_en_inferencia_df_modelo = [col for col in features_modelo_final if col not in columnas_modelo]
extras_en_inferencia_df = [col for col in columnas_inferencia if col not in features_modelo_final]

print(f'Columnas visibles en inferencia_df: {len(columnas_inferencia)}')
print(f'Variables esperadas por el modelo final: {len(features_modelo_final)}')
print(f'Columnas en inferencia_df_modelo: {len(columnas_modelo)}')
print(f'Variables del modelo que faltan en inferencia_df: {len(faltan_en_inferencia_df)}')
print(f'Variables del modelo que faltan en inferencia_df_modelo: {len(faltan_en_inferencia_df_modelo)}')

if faltan_en_inferencia_df:
    print('Primeras variables que no aparecen en inferencia_df pero s? en el modelo:')
    print(faltan_en_inferencia_df[:20])

if extras_en_inferencia_df:
    print('Columnas descriptivas presentes en inferencia_df que no usa el modelo:')
    print(extras_en_inferencia_df)

if not faltan_en_inferencia_df_modelo:
    print('Verificaci?n correcta: inferencia_df_modelo est? completamente alineado con el modelo final.')
else:
    print('Atenci?n: todav?a faltan variables en inferencia_df_modelo.')
